# Derived outputs

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/03-derived-outputs` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

A model run gives you the number of people in each compartment on each day.
Usually you want something else: how many people are infectious in total, how many
new infections there are each day, how many people have been infected so far, or what
share of the population is infectious. These are *derived outputs*.

This page covers, on one small SIR model:

- an `OutputSet`, a named list of the outputs you want, and how to refer to one output
  from another with `outputs.ref`;
- compartment sizes with `Compartments`, and flow rates such as incidence with `FlowMass`;
- adding up groups of compartments with `sum_by`;
- counting people over time: `cumulative_integral()`, `integrate_intervals().cumsum()` and
  `integrate()`;
- reading one day's value with `at(t)`;
- ratios (prevalence), NumPy functions on an output, and going out to a plain array and back
  with `array()` and `with_values()`.

## The model

An SIR model with 1,000 people, 10 of them infectious on day 0. The two flows are
infection (driven by the number of infectious people and the contact rate `beta`) and
recovery (at rate `gamma`).

The next cell builds and compiles the model. It draws nothing.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowMass,
    FlowModel,
    OutputSet,
    Param,
    Property,
    PropertyMap,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection

state = Property("state", ("S", "I", "R"))
model = FlowModel(PropertyMap.from_property(state))
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection("foi", infectious=state["I"], contact_rate=Param("beta")),
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("gamma")))
model.set_initial_population({state["S"]: 990.0, state["I"]: 10.0, state["R"]: 0.0})
cm = model.compile()
params = {"beta": 0.4, "gamma": 0.1}

## Naming the outputs you want

An `OutputSet` is a dictionary of the outputs you want. You write each one down as a
recipe, and nothing is calculated yet:

- `Compartments()` is the number of people in every compartment on every day, and
  `Compartments(where=state["I"])` keeps only the infectious ones.
- `.sum_by()` adds the compartments up, so you get one number per day. Without any
  argument it adds up everything; the next page shows how to keep strata apart.
- `outputs.ref("name")` stands for an output you already defined, so one output can be
  built from another. Dividing infectious people by the population gives prevalence.

The model is then solved once with `outputs.plan()`, which tells the solver what to save,
and `outputs.evaluate(...)` turns the recipes into numbers. The plot shows the number of
people who are infectious, rising to a peak and falling again.

In [ ]:
outputs = OutputSet()
outputs["population"] = Compartments().sum_by()
outputs["infectious"] = Compartments(where=state["I"]).sum_by()
outputs["prevalence"] = outputs.ref("infectious") / outputs.ref("population")

result = cm.run(params, t0=0.0, t1=100.0, dt=1.0, save=outputs.plan())
named = outputs.evaluate(result, params)

named.to_frame(names=["infectious"], backend="pandas").plot(
    title="Infectious people",
    labels={"time": "day", "value": "people", "variable": ""},
)

## Prevalence as a ratio of outputs

Because `prevalence` was defined as one output divided by another, it is itself an output
with a value on every day. Here it is as a percentage of the population. The curve should
have the same shape as the one above, with a peak of about 43% around day 21.

In [ ]:
(100 * named["prevalence"]).to_pandas().rename(columns={"value": "prevalence"}).plot(
    title="Prevalence of infection",
    labels={"time": "day", "value": "% of population", "variable": ""},
)

## Flow rates: incidence

`FlowMass("infection")` is the rate at which people move along the infection flow, in
people per day, saved on every day. The flow is saved once for every pair of compartments it
connects, and `.sum_by()` adds those up to a single daily rate. The plot should rise to
a peak a few days before the peak in the number of infectious people, because new infections
are what feed it.

In [ ]:
outputs["incidence"] = FlowMass("infection").sum_by()

result = cm.run(params, t0=0.0, t1=100.0, dt=1.0, save=outputs.plan())
named = outputs.evaluate(result, params)

named.to_frame(names=["incidence"], backend="pandas").plot(
    title="Incidence: new infections per day",
    labels={"time": "day", "value": "people per day", "variable": ""},
)

## Counting people over time

A flow output is a *rate* (people per day). To count people you add the rate up over
time, and there are three ways to do it:

- `cumulative_integral()` gives the running total of a rate: the number of people infected
  from day 0 up to each day. It starts at 0.
- `integrate_intervals()` gives the number of people in each interval between two saved days,
  and `cumsum()` adds those amounts up. Use `cumsum()` on amounts like these, not on a rate.
  Because it describes intervals, the result has one value fewer than there are saved days,
  and each value is stamped with the end of its interval.
- `integrate()` gives a single number, the total over the whole run.

The two running totals should lie on the same line. Adding up a rate with `cumsum()` would
be wrong whenever the saved days are not exactly one day apart.

In [ ]:
outputs["cumulative_infections"] = outputs.ref("incidence").cumulative_integral()
outputs["new_per_interval"] = outputs.ref("incidence").integrate_intervals()
outputs["cumulative_from_intervals"] = outputs.ref("new_per_interval").cumsum()
outputs["total_infections"] = outputs.ref("incidence").integrate()

result = cm.run(params, t0=0.0, t1=100.0, dt=1.0, save=outputs.plan())
named = outputs.evaluate(result, params)

both = pd.concat(
    {
        "cumulative_integral()": named["cumulative_infections"].to_pandas()["value"],
        "integrate_intervals().cumsum()": named["cumulative_from_intervals"].to_pandas()["value"],
    },
    axis=1,
)
both.plot(
    title="Cumulative infections, two ways",
    labels={"time": "day", "value": "people", "variable": ""},
)

`integrate()` returned a single number, shown below with `.scalar()`. In this closed model every
infection takes one person out of S, so the total should be close to the 990 people who
were susceptible at the start; a few susceptible people are left at the end.

In [ ]:
float(named["total_infections"].scalar())

## The value on one day: `at(t)`

`at(t)` picks out the value at day `t`. Subtracting it from a whole series gives the change
since that day. The plot shows the change in the number of infectious people since day 30,
so it is zero on day 30 itself. It starts well below zero (few people were infectious on
day 0), is positive around the peak, and is negative after day 30 as the epidemic dies away.

In [ ]:
outputs["change_since_30"] = outputs.ref("infectious") - outputs.ref("infectious").at(30.0)

result = cm.run(params, t0=0.0, t1=100.0, dt=1.0, save=outputs.plan())
named = outputs.evaluate(result, params)

named.to_frame(names=["change_since_30"], backend="pandas").plot(
    title="Change in infectious people since day 30",
    labels={"time": "day", "value": "people", "variable": ""},
)

## NumPy functions on an output

Ordinary NumPy functions work on an output, and give another output. Here `np.log` of the
prevalence is plotted. While the epidemic grows exponentially this is close to a straight line,
whose slope is the growth rate (a little under `beta - gamma` = 0.3 per day at first); the
line then bends over as susceptibles run out.

In [ ]:
outputs["log_prevalence"] = np.log(outputs.ref("prevalence"))

result = cm.run(params, t0=0.0, t1=100.0, dt=1.0, save=outputs.plan())
named = outputs.evaluate(result, params)

named.to_frame(names=["log_prevalence"], backend="pandas").plot(
    title="Natural log of prevalence",
    labels={"time": "day", "value": "log(prevalence)", "variable": ""},
)

## Out to a plain array and back

For anything an output cannot do directly, `array()` gives its values as a plain array, with
no times attached. You can do whatever you like to the array, and `with_values()` gives the
result the output's days again so it can be plotted or compared with data. Here the
prevalence is expressed relative to its own peak, so the highest point is exactly 1.

In [ ]:
prevalence = named["prevalence"]
values = prevalence.array()
relative = prevalence.with_values(values / values.max())

relative.to_pandas().rename(columns={"value": "relative prevalence"}).plot(
    title="Prevalence relative to its peak",
    labels={"time": "day", "value": "fraction of peak", "variable": ""},
)

## Things to try

- Change `beta` in `params` and rerun the cells from "Naming the outputs you want". How do the
  peak of the prevalence and the total number of infections change?
- Add `outputs["recovered"] = Compartments(where=state["R"]).sum_by()` and plot it against
  the cumulative infections. What is the gap between the two lines?
- Replace `.at(30.0)` with another day, or divide the incidence by `.integrate()` of itself to
  get the share of all infections that happen each day.
- Ask for `FlowMass("recovery").sum_by().cumulative_integral()` and compare it with the
  number of recovered people.